# secfsn_plus_models: from prepared evidence to model handoff

**Layer 04 of the SECFSN stack** turns a validated company-quarter feature panel into evaluated model signals for a downstream decision system.

- Consumes prepared financial-statement-derived features—not raw SEC filings.
- Builds forward-return labels, preserves time order, and evaluates held-out predictions.
- Packages an auditable `(cik, period)` model-output contract for Layer 05.

> This recording uses the repository's deterministic **synthetic** fixture. It demonstrates engineering flow, not investment advice or a live SEC-data result.

In [1]:
from pathlib import Path
import warnings

import pandas as pd

from secfsn_plus_models.notebook_entry import init_notebook

init_notebook()


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "secfsn_plus_models").is_dir():
            return candidate
    raise RuntimeError("Run this notebook from the repository root or a descendant directory.")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 10)
pd.set_option("display.width", 110)

from secfsn_plus_models.automation import ResolvedConfig, build_pipeline_artifacts, load_feature_matrix
from secfsn_plus_models.evaluation import evaluate_cross_sectional_model
from secfsn_plus_models.outputs import build_model_output_frame


Notebook environment initialized.


## 1. Representative input

In production, Layers 02–03 turn SEC financial-statement evidence into this canonical panel. This repository receives the prepared features plus optional metadata such as price and sector.

For a repeatable recording, we deliberately use the built-in seeded fixture; no credentials, internet access, or private data are involved.

In [2]:
config = ResolvedConfig(
    price_column="price",
    train_end_period="2020Q4",
    model_type="linear_regression",
    test_size=None,
    use_synthetic_data=True,
    data_path=None,
    feature_matrix_path=None,
    synthetic_seed=20240323,
    robustness_runs=10,
)

features_df, dataset_source = load_feature_matrix(
    price_column=config.price_column,
    use_synthetic_data=config.use_synthetic_data,
    synthetic_seed=config.synthetic_seed,
)

input_preview = (
    features_df.reset_index()
    .loc[:, ["cik", "period", "feature_growth", "feature_quality", "feature_stability", "price"]]
    .head(6)
)

print(f"Input source: {dataset_source}")
print(f"Canonical index: {features_df.index.names} | {len(features_df)} company-quarter rows")
input_preview.style.format({
    "feature_growth": "{:.3f}",
    "feature_quality": "{:.3f}",
    "feature_stability": "{:.3f}",
    "price": "{:.2f}",
})


Input source: generated synthetic fallback dataset (seed=20240323)
Canonical index: ['cik', 'period'] | 42 company-quarter rows


,cik,period,feature_growth,feature_quality,feature_stability,price
0,0001,2020Q1,0.653,-0.041,0.078,92.24
1,0001,2020Q2,0.308,-0.074,0.001,92.88
2,0001,2020Q3,1.030,-0.041,0.305,94.17
3,0001,2020Q4,-0.247,-0.203,0.116,93.38
4,0001,2021Q1,0.800,-0.032,0.544,92.92
5,0001,2021Q2,-0.224,-0.090,0.233,96.76


## 2. Production workflow and result

One repository-native call validates the panel, creates correctly aligned forward labels, separates metadata, makes a time-based split, trains the registered model, and produces held-out predictions. The final adapter turns those predictions into the Layer 05 handoff shape.

In [3]:
artifacts = build_pipeline_artifacts(config, features_df=features_df)
model_outputs = build_model_output_frame(
    artifacts.predictions,
    higher_prediction_more_risky=False,  # lower expected return maps to higher ranked risk
)
evaluation = evaluate_cross_sectional_model(artifacts.y_test, artifacts.predictions)

assert model_outputs.index.equals(artifacts.predictions.index)

workflow_summary = pd.DataFrame(
    [
        {"stage": "Validated input", "evidence": f"{len(artifacts.features_df)} rows / {len(artifacts.X.columns)} modeled features"},
        {"stage": "Forward labels", "evidence": f"{len(artifacts.labels)} next-period returns; final period excluded"},
        {"stage": "Time split", "evidence": f"{len(artifacts.X_train)} train rows through 2020Q4; {len(artifacts.X_test)} held-out rows"},
        {"stage": "Handoff contract", "evidence": "prediction index preserved exactly for Layer 05"},
    ]
)
display(workflow_summary)

evaluation_display = pd.DataFrame(
    {
        "value": evaluation,
        "interpretation": [
            "Mean Pearson cross-sectional correlation by held-out period",
            "Mean rank correlation by held-out period",
            "Not available: each held-out period has 6 issuers; period deciles need at least 10 unique predictions",
        ],
    }
)
display(evaluation_display.style.format({"value": "{:.3f}"}, na_rep="—"))

handoff_preview = (
    model_outputs.reset_index()
    .sort_values(["period", "model_risk_probability"], ascending=[True, False])
    .head(6)
)
display(
    handoff_preview.style.format({
        "model_prediction": "{:.3f}",
        "model_expected_return": "{:.3%}",
        "model_risk_probability": "{:.0%}",
    })
)


,stage,evidence
0,Validated input,42 rows / 5 modeled features
1,Forward labels,36 next-period returns; final period excluded
2,Time split,24 train rows through 2020Q4; 12 held-out rows
3,Handoff contract,prediction index preserved exactly for Layer 05


,value,interpretation
IC,0.088,Mean Pearson cross-sectional correlation by held-out period
RankIC,0.143,Mean rank correlation by held-out period
DecileSpread,—,Not available: each held-out period has 6 issuers; period deciles need at least 10 unique predictions


,cik,period,model_prediction,model_expected_return,model_risk_probability
4,0005,2021Q1,-0.024,-2.421%,100%
2,0003,2021Q1,-0.016,-1.579%,83%
5,0006,2021Q1,-0.007,-0.662%,67%
1,0002,2021Q1,0.004,0.354%,50%
3,0004,2021Q1,0.011,1.073%,33%
0,0001,2021Q1,0.017,1.676%,17%


## Takeaway

This demonstrated a repeatable Layer 04 path from an upstream company-quarter feature panel to evaluated, index-aligned signals for `secfsn_plus_decisionsystems`.

- **Reproducible:** a seeded local fixture and deterministic linear model.
- **Auditable:** explicit `(cik, period)` contracts, forward-label alignment, and time separation.
- **Bounded:** risk probability is a ranked output adapter—not calibrated risk, investment advice, or a final decision.

Raw SEC ingestion, accounting-ratio construction, structural diagnostics, and decision rules remain in the other SECFSN layers.